In [ ]:
from skimage.io import imread
from skimage.filters import frangi
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, remove_small_objects
from scipy.ndimage import distance_transform_edt
from Skel2Graph3D import skel2graph3d
import scipy.io as sio


py_bin = sio.loadmat('/Users/mutrera/Documents/Uchicago College/Uchicago 2024 - 2025/Summer/Octava/octava/code/Skeletonization/matlab_skels/binarise_python_result.mat')
py_bin = py_bin['Y_py']
skel = skeletonize(py_bin)

node, link = skel2graph3d(skel, 1)

# Quick test to check Skeletonize func is equal to matlab

In [ ]:
skel_func_res = skeletonize(adpt_mask)

In [6]:
len(node)

5307

In [7]:
len(link)

4546

In [ ]:
from scipy.io import savemat
savemat('skel_func_res.mat', {'my_array': skel_func_res})

# Fun experiment to see if GPT could translate the entire thing

I have not tested this, but who knows? What if it solves all of my problems

In [ ]:
import numpy as np
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize, remove_small_objects
import networkx as nx

def skeletonize_main(bin_img, THR=1):
    """
    Input: bin_img (2D numpy array, boolean or 0/1)
           THR (int) minimum branch length (in pixels) to keep
    Returns:
      skel (2D boolean numpy array) - skeleton after pruning
      nodes (list of dict) - node structures similar to MATLAB (idx, links, conn, comx, comy, ep, label)
      links (list of dict) - link structures (n1, n2, point[list of linear indices])
      D (2D float array) - local thickness map (approx) = 2*distance transform of foreground
    """
    # ensure boolean
    L = (bin_img > 0)
    # skeletonize (2D)
    sk = skeletonize(L)
    # optional: remove small isolated skeleton components (component-size)
    # remove components smaller than THR (in pixels) from skeleton
    if THR > 1:
        sk = remove_small_objects(sk, min_size=THR, connectivity=2)
    # local thickness map (distance transform of foreground gives distance to background)
    D = distance_transform_edt(L) * 2.0

    # Build graph from skeleton: nodes = pixels with degree != 2 (endpoints or junctions)
    nodes, links = skel_to_graph_2d(sk, THR=THR)

    # optionally: rebuild skeleton from nodes+links to ensure consistency (like Graph2Skel3D)
    sk_rebuilt = graph_to_skel(nodes, links, sk.shape)
    # re-run node/link extraction until stable (optional; guard with max_iter)
    wl = sum(len(l['point']) for l in links)
    max_iter = 10
    for _ in range(max_iter):
        nodes, links = skel_to_graph_2d(sk_rebuilt, THR=0)  # get full graph
        sk_rebuilt = graph_to_skel(nodes, links, sk.shape)
        wl_new = sum(len(l['point']) for l in links)
        # stop if change < 0.5% or exactly stable
        if wl == 0:
            if wl_new == 0:
                break
        else:
            if abs(wl_new - wl) / max(1, wl) < 0.005:
                break
        wl = wl_new

    # After final cleanup, prune short links using THR (if not done already)
    # (skel_to_graph_2d already pruned when THR>0)
    # ensure boolean dtype
    return sk_rebuilt.astype(bool), nodes, links, D


# ------------------------
# Helper: get 8-neighbour indices
# ------------------------
_neigh_offsets = [(-1,-1), (-1,0), (-1,1),
                  ( 0,-1),         ( 0,1),
                  ( 1,-1), ( 1,0), ( 1,1)]

def _neighbors(y, x, shape):
    h, w = shape
    for dy, dx in _neigh_offsets:
        ny, nx = y + dy, x + dx
        if 0 <= ny < h and 0 <= nx < w:
            yield ny, nx

# ------------------------
# Convert skeleton to graph (2D)
# ------------------------
def skel_to_graph_2d(skel, THR=1):
    """
    Build graph nodes & links from 2D skeleton.
    A node is any skeleton pixel with degree != 2 (junction or endpoint).
    Links are chains of degree-2 pixels between nodes.

    Returns:
      nodes: list of dicts { 'idx': [linear indices of node pixels],
                             'links': [link_ids],
                             'conn': [connected_node_ids],
                             'comx','comy', 'ep' }
      links: list of dicts { 'n1': node_id, 'n2': node_id, 'point': [linear indices chain] }
    THR: minimum length to keep a link (in pixels). links shorter than THR are discarded.
    """
    H, W = skel.shape
    coords = np.argwhere(skel)            # N x 2 array of (y,x)
    idx_map = -np.ones(skel.shape, dtype=int)
    for i,(y,x) in enumerate(coords):
        idx_map[y,x] = i

    # compute degree for each skeleton pixel (8-connectivity)
    deg = np.zeros(len(coords), dtype=int)
    for i,(y,x) in enumerate(coords):
        cnt = 0
        for ny,nx in _neighbors(y,x, skel.shape):
            if skel[ny,nx]:
                cnt += 1
        deg[i] = cnt

    # Identify nodes (deg != 2) and canal pixels (deg == 2) and endpoints (deg == 1)
    node_pixel_mask = (deg != 2)
    node_pixels_idx = np.where(node_pixel_mask)[0]
    canal_pixels_idx = np.where(deg == 2)[0]

    # Map pixel index to node-id (for node pixels only)
    pixelidx_to_nodeid = -np.ones(len(coords), dtype=int)
    for n_id, pix_idx in enumerate(node_pixels_idx):
        pixelidx_to_nodeid[pix_idx] = n_id

    nodes = []
    for n_id, pix_idx in enumerate(node_pixels_idx):
        y,x = coords[pix_idx]
        # treat single-pixel node cluster (we don't cluster multi-pixel nodes)
        nodes.append({
            'idx': [np.ravel_multi_index((y,x),(H,W))],
            'links': [],
            'conn': [],
            'comx': x,
            'comy': y,
            'ep': 1 if deg[pix_idx] == 1 else 0,
            'label': 0
        })

    # We'll walk from each node out along each neighbor that is a canal pixel
    links = []
    link_id = 0
    visited = set()  # visited pixel linear indices in coords-space (use coords index)
    # helper: convert linear position within coords -> (y,x)
    def coords_pos_to_yx(pos_index):
        return tuple(coords[pos_index])

    # mapping from pixel coords index -> coords position (i.e., index into coords array) is just pos
    for n_local_idx, pix_idx in enumerate(node_pixels_idx):
        # pixel coordinates
        y0, x0 = coords[pix_idx]
        # explore neighbors
        for ny,nx in _neighbors(y0,x0, skel.shape):
            if not skel[ny,nx]:
                continue
            neigh_pos = idx_map[ny,nx]
            if neigh_pos == -1:
                continue
            # If neighbor is a node (deg!=2), then create a direct link of length 1
            if pixelidx_to_nodeid[neigh_pos] != -1:
                neighbor_node_id = pixelidx_to_nodeid[neigh_pos]
                if neighbor_node_id == n_local_idx:
                    continue
                # create trivial link if not already present
                # ensure we don't duplicate by storing symmetric check
                already = False
                for L in nodes[n_local_idx]['links']:
                    l = links[L]
                    if (l['n1']==n_local_idx and l['n2']==neighbor_node_id) or (l['n2']==n_local_idx and l['n1']==neighbor_node_id):
                        already = True
                        break
                if not already:
                    links.append({'n1': n_local_idx, 'n2': neighbor_node_id,
                                  'point': [np.ravel_multi_index((y0,x0),(H,W)), np.ravel_multi_index((ny,nx),(H,W))]})
                    nodes[n_local_idx]['links'].append(link_id)
                    nodes[n_local_idx]['conn'].append(neighbor_node_id)
                    nodes[neighbor_node_id]['links'].append(link_id)
                    nodes[neighbor_node_id]['conn'].append(n_local_idx)
                    link_id += 1
                continue

            # else neighbor is canal pixel -> follow it until we hit a node
            chain = []
            cur_pos = neigh_pos
            prev_pos = pix_idx
            chain.append(np.ravel_multi_index((ny,nx),(H,W)))
            # walk until you reach a node pixel
            while True:
                # if visited this coords position earlier in this walk, break to avoid loops
                if cur_pos == -1:
                    break
                ycur, xcur = coords[cur_pos]
                # find next neighbors among skeleton (excluding previous)
                nexts = []
                for nny, nnx in _neighbors(ycur, xcur, skel.shape):
                    if not skel[nny,nnx]:
                        continue
                    posn = idx_map[nny,nnx]
                    if posn == -1 or posn == prev_pos:
                        continue
                    nexts.append(posn)
                # if neighbor is node (deg != 2), finish
                if pixelidx_to_nodeid[cur_pos] != -1:
                    end_node_id = pixelidx_to_nodeid[cur_pos]
                    # record final node pixel coordinate as part of chain too
                    # chain already contains intermediates; we may append the node pixel coords
                    # but for consistency with MATLAB, link.points often exclude node pixels outermost; we will include both ends
                    chain.append(np.ravel_multi_index(tuple(coords[cur_pos]), (H,W)))
                    # if the chain is long enough, add link (length measured as number of pixels in chain)
                    if len(chain) >= THR:
                        links.append({'n1': n_local_idx, 'n2': end_node_id, 'point': chain.copy()})
                        # update nodes connections if not duplicate
                        nodes[n_local_idx]['links'].append(link_id)
                        nodes[n_local_idx]['conn'].append(end_node_id)
                        nodes[end_node_id]['links'].append(link_id)
                        nodes[end_node_id]['conn'].append(n_local_idx)
                        link_id += 1
                    break
                # if no next (dead end), break
                if len(nexts) == 0:
                    break
                # otherwise step to the next canal pixel (prefer the first)
                prev_pos = cur_pos
                cur_pos = nexts[0]
                ycur, xcur = coords[cur_pos]
                chain.append(np.ravel_multi_index((ycur,xcur),(H,W)))
                # safety: if chain gets very long, exit (shouldn't happen)
                if len(chain) > (H*W):
                    break

    # remove potential duplicate links (same n1/n2 and same chain reversed)
    canonical = {}
    clean_links = []
    for l in links:
        a,b = min(l['n1'], l['n2']), max(l['n1'], l['n2'])
        key = (a,b, tuple(l['point']))  # chain included to avoid merging distinct parallel chains
        if key not in canonical:
            canonical[key] = True
            clean_links.append(l)
    links = clean_links

    # finalize nodes: unique conn and links
    for n in nodes:
        n['conn'] = list(dict.fromkeys(n['conn']))
        n['links'] = list(dict.fromkeys(n['links']))

    # If no node found, return empty node/link structures
    if len(nodes) == 0:
        nodes = []
        links = []

    return nodes, links

# ------------------------
# Rebuild skeleton from graph
# ------------------------
def graph_to_skel(nodes, links, shape):
    """
    Rebuild binary skeleton image from nodes+links.
    nodes: list of node dicts with 'idx' as list of linear indices
    links: list of link dicts with 'point' list of linear indices
    """
    H,W = shape
    sk = np.zeros((H,W), dtype=bool)
    for n in nodes:
        for lin in n['idx']:
            y,x = np.unravel_index(lin, (H,W))
            sk[y,x] = True
    for l in links:
        for lin in l['point']:
            y,x = np.unravel_index(lin, (H,W))
            sk[y,x] = True
    return sk
